# Mesh generation and visualization
*Authors: Corrado Maurini (corrado.maurini@sorbonne-universite.fr)*

In this notebook you will find examples to
 - Generate meshes with `gmsh` and import them in `dolfinx`
 - Visualize the `dolfinx` mesh directly in the notebook using `pyvista`
 - Save the mesh to an `xdmf` file

We consider an elastic slab $\Omega$ with a straight crack $\Gamma$
Using the symmetry, we will consider only half of the domain in the computation and we refine the mesh around the crack tip.

![title](domain.png)


Let us generate a mesh using gmsh (http://gmsh.info/).

The function to generate the mesh is reported in the external file `meshes.py`.
The mesh is refined around the crack tip.

In [ ]:
import gmsh
import numpy as np
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio
import dolfinx.plot

We define below the geometrical parameters of the mesh.

In [ ]:
Lx = 1.0
Ly = 0.5
Lcrack = 0.3
lc = 0.1
refinement_ratio = 10
graded = True
grade = 5.0
dist_min = 0.1
dist_max = 0.3
gdim = 2


The following code block generates the mesh using the `gmsh` python interface. We refer the reader to the `gmsh` documentation for details: https://gmsh.info/doc/texinfo/gmsh.html

We also define *MPI communicator*s, which are required only in parallel computation.
They are not strictly necessary here, but we keep them to have this example working in general.
The are set so that when we perform parallel computations, the mesh is generated on one processor (`model_rank=0`) and then it is distributed to all the processors (`mesh_comm = MPI.COMM_WORLD`). Further documentation about the use of MPI can be found at https://scientificcomputing.github.io/mpi-tutorial/notebooks/dolfinx_MPI_tutorial.html

We define a dictionary `tags` to associate clear names to numerical tags that are used to identify the different parts of the domain and the boundary

The element size is set by a *field*, a function of the position evaluated by
`gmsh` as it meshes. Around the tip it follows the distance $r$ to it,

$$h(r) = \min\left(l_c,\ \max\left(h_\mathrm{min},\ \frac{r}{g}\right)\right),
\qquad h_\mathrm{min} = \frac{l_c}{\texttt{refinement\_ratio}}.$$

The displacement varies as $\sqrt{r}$ near the tip, so the mesh has to be
self-similar there: wherever the floor $h_\mathrm{min}$ is inactive the ratio
$h/r$ is the constant $1/g$, and the rings of elements around the tip all have
the same shape. The floor fixes the size of the elements touching the tip.

With `graded = False` the size is instead uniform and equal to
$h_\mathrm{min}$ over the disc $r < \texttt{dist\_min}$, then grows linearly
up to $l_c$, reached at $r = \texttt{dist\_max}$. That is the `Threshold`
field of the `gmsh` tutorials. For a given accuracy on the energy it costs
several times more elements than the graded field, since the uniform disc
carries small elements where the solution does not need them.

In [ ]:
mesh_comm = MPI.COMM_WORLD
model_rank = 0
gmsh.initialize()

facet_markers = {"left": 1, "right": 2, "top": 3, "crack": 4, "bottom_no_crack": 5}
cell_markers = {"all": 20}
point_markers = {"crack_tip": 10}
h_min = lc / refinement_ratio

# every process needs a model to hand to `model_to_mesh` below, but only
# `model_rank` builds the geometry and meshes it
model = gmsh.model()
if mesh_comm.rank == model_rank:
    model.add("Rectangle")
    model.setCurrent("Rectangle")
    # Create the points
    p1 = model.geo.addPoint(0.0, 0.0, 0, lc)
    p2 = model.geo.addPoint(Lcrack, 0.0, 0, lc)
    p3 = model.geo.addPoint(Lx, 0, 0, lc)
    p4 = model.geo.addPoint(Lx, Ly, 0, lc)
    p5 = model.geo.addPoint(0, Ly, 0, lc)
    # Create the lines
    l1 = model.geo.addLine(p1, p2, tag=facet_markers["crack"])
    l2 = model.geo.addLine(p2, p3, tag=facet_markers["bottom_no_crack"])
    l3 = model.geo.addLine(p3, p4, tag=facet_markers["right"])
    l4 = model.geo.addLine(p4, p5, tag=facet_markers["top"])
    l5 = model.geo.addLine(p5, p1, tag=facet_markers["left"])
    # Create the surface
    cloop1 = model.geo.addCurveLoop([l1, l2, l3, l4, l5])
    surface_1 = model.geo.addPlaneSurface([cloop1])

    # Field 1 gives the distance to the crack tip
    model.mesh.field.add("Distance", 1)
    model.mesh.field.setNumbers(1, "PointsList", [p2])
    # Field 2 turns that distance into an element size
    if graded:
        #  Size
        #     lc -                        /---------------
        #                                /
        # SizeMin -o------------------- /
        #          |                   |
        #          0          grade * SizeMin          Distance
        model.mesh.field.add("MathEval", 2)
        model.mesh.field.setString(2, "F", f"Min({lc}, Max({h_min}, F1 / {grade}))")
    else:
        # SizeMax -                   / ------------------
        #                            /
        # SizeMin -o----------------/
        #          |                |  |
        #        Point        DistMin   DistMax
        model.mesh.field.add("Threshold", 2)
        model.mesh.field.setNumber(2, "InField", 1)
        model.mesh.field.setNumber(2, "SizeMin", h_min)
        model.mesh.field.setNumber(2, "SizeMax", lc)
        model.mesh.field.setNumber(2, "DistMin", dist_min)
        model.mesh.field.setNumber(2, "DistMax", dist_max)
    model.mesh.field.setAsBackgroundMesh(2)
    # the field alone sets the size: the sizes carried by the points of the
    # geometry would otherwise compete with it
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("General.Verbosity", 3)
    model.geo.synchronize()

    # Assign the physical groups, before meshing. The name of a group is what
    # `dolfinx` reports in `physical_groups`, the tag is what the meshtags carry
    model.addPhysicalGroup(2, [surface_1], tag=cell_markers["all"])
    model.setPhysicalName(2, cell_markers["all"], "Rectangle surface")

    for key, value in facet_markers.items():
        model.addPhysicalGroup(1, [value], tag=value)
        model.setPhysicalName(1, value, key)

    # the tip is a group of its own, so that it can be found without a
    # geometrical search
    model.addPhysicalGroup(0, [p2], tag=point_markers["crack_tip"])
    model.setPhysicalName(0, point_markers["crack_tip"], "crack_tip")

    model.mesh.generate(gdim)


We can now import the mesh in `dolfinx`

In [ ]:
msh, cell_tags, facet_tags, *_ = gmshio.model_to_mesh(
    model, mesh_comm, model_rank, gdim=gdim
)
gmsh.finalize()
msh.name = "rectangle"
cell_tags.name = f"{msh.name}_cells"
facet_tags.name = f"{msh.name}_facets"

In [ ]:

with dolfinx.io.XDMFFile(MPI.COMM_WORLD, "output/mesh.xdmf", "w") as file:
    file.write_mesh(msh)
    msh.topology.create_connectivity(1, 2)
    file.write_meshtags(cell_tags, msh.geometry)
    file.write_meshtags(facet_tags, msh.geometry)

To plot the mesh we use `pyvista` see:
- https://jorgensd.github.io/dolfinx-tutorial/chapter3/component_bc.html
- https://docs.fenicsproject.org/dolfinx/main/python/demos/pyvista/demo_pyvista.py.html

In [ ]:
import pyvista

pyvista.set_jupyter_backend("static")


# Extract topology from mesh and create pyvista mesh
topology, cell_types, x = dolfinx.plot.vtk_mesh(msh)
grid = pyvista.UnstructuredGrid(topology, cell_types, x)

plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True)
plotter.camera_position = "xy"

if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    plotter.screenshot("mesh.png")

We wrap the code to generate the mesh in the external module `../utils/meshes` to reuse it in the following tutorials.
We can use it as follows:

In [ ]:
import sys

sys.path.append("../utils")
from meshes import generate_mesh_with_crack

msh, mt, ft = generate_mesh_with_crack(
    Lcrack=Lcrack,
    Ly=0.5,
    lc=0.1,  # caracteristic length of the mesh
    refinement_ratio=10,  # how much it is refined at the tip zone
    graded=True,  # size growing with the distance to the tip
    grade=5.0,  # slope r/h of the graded field
    dist_min=dist_min,  # radius of tip zone, used only if graded is False
    dist_max=dist_max,  # radius of the transition zone, idem
    verbosity=5,
)
